# Olist Sellers Agent

Foco exclusivo na análise dos sellers como clientes da Olist, com métricas de receita, ticket médio, frete e performance por categoria.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore')

## Conectar o Google Drive

Para rodar no Colab, precisamos montar o Drive para acessar os CSVs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Carregar dados relevantes dos sellers

In [ ]:
base_path = '/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/'
orders = pd.read_csv(base_path + 'olist_orders_dataset.csv')
order_items = pd.read_csv(base_path + 'olist_order_items_dataset.csv')
products = pd.read_csv(base_path + 'olist_products_dataset.csv')
sellers = pd.read_csv(base_path + 'olist_sellers_dataset.csv')
payments = pd.read_csv(base_path + 'olist_order_payments_dataset.csv')
product_category = pd.read_csv(base_path + 'product_category_name_translation.csv')
sellers['seller_label'] = 'seller_' + sellers['seller_id'].astype(str)

print('Shapes:')
print('orders', orders.shape)
print('order_items', order_items.shape)
print('products', products.shape)
print('payments', payments.shape)
print('sellers', sellers.shape)

## Preparar a tabela mestre de sellers

In [ ]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_status'] = orders['order_status'].astype('category')

items_products = pd.merge(
    order_items,
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)
items_products = pd.merge(
    items_products,
    sellers[['seller_id', 'seller_label', 'seller_zip_code_prefix', 'seller_city', 'seller_state']],
    on='seller_id',
    how='left'
)

seller_master = pd.merge(
    items_products,
    orders[[
        'order_id',
        'customer_id',
        'order_status',
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_customer_date'
    ]],
    on='order_id',
    how='left'
)

seller_master = pd.merge(
    seller_master,
    payments[['order_id', 'payment_type', 'payment_installments', 'payment_value']],
    on='order_id',
    how='left'
)

seller_master = pd.merge(
    seller_master,
    product_category,
    on='product_category_name',
    how='left'
)

seller_master['delivered'] = seller_master['order_status'] == 'delivered'
seller_master['price'] = pd.to_numeric(seller_master['price'], errors='coerce')
seller_master['freight_value'] = pd.to_numeric(seller_master['freight_value'], errors='coerce')
seller_master['payment_value'] = pd.to_numeric(seller_master['payment_value'], errors='coerce')

seller_metrics = (
    seller_master[seller_master['delivered']]
    .groupby(['seller_label'])
    .agg(
        revenue=('price', 'sum'),
        order_count=('order_id', 'nunique'),
        total_freight=('freight_value', 'sum'),
        total_payment=('payment_value', 'sum')
    )
    .reset_index()
)

seller_metrics['avg_order_value'] = seller_metrics['revenue'] / seller_metrics['order_count']
seller_metrics['avg_freight'] = seller_metrics['total_freight'] / seller_metrics['order_count']
seller_metrics['avg_payment_value'] = seller_metrics['total_payment'] / seller_metrics['order_count']
seller_metrics['freight_ratio'] = seller_metrics['total_freight'] / seller_metrics['revenue']
seller_metrics['revenue_per_order'] = seller_metrics['revenue'] / seller_metrics['order_count']

seller_metrics.head(10)

## Top sellers por receita

In [ ]:
top_sellers = seller_metrics.sort_values('revenue', ascending=False).head(10)
plt.figure(figsize=(12, 6))
sns.barplot(data=top_sellers, x='revenue', y='seller_label', palette='coolwarm')
plt.title('Top 10 sellers por receita total')
plt.xlabel('Receita total (R$)')
plt.ylabel('Seller')
plt.tight_layout()
plt.show()

## Métricas de eficiência de sellers

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=seller_metrics, x='avg_order_value', y='avg_freight', size='order_count', legend=False, alpha=0.8)
plt.title('Ticket médio x frete médio por seller')
plt.xlabel('Ticket médio (R$)')
plt.ylabel('Frete médio (R$)')
plt.tight_layout()
plt.show()

## Categorias de produto relevantes por seller

In [ ]:
seller_category = (
    seller_master[seller_master['delivered']]
    .groupby(['seller_label', 'product_category_name'])
    .agg(revenue=('price', 'sum'), order_count=('order_id', 'nunique'))
    .reset_index()
)

category_revenue = (
    seller_category.groupby('product_category_name')
    .agg(revenue=('revenue', 'sum'), order_count=('order_count', 'sum'))
    .reset_index()
    .sort_values('revenue', ascending=False)
)


top_categories = category_revenue.head(10)top_seller_categories.head(10)



plt.figure(figsize=(12, 6)))

sns.barplot(data=top_categories, x='revenue', y='product_category_name', palette='viridis')    .head(3)

plt.title('Top 10 categorias por receita total')    .groupby('seller_label')

plt.xlabel('Receita total (R$)')    seller_category_share.sort_values(['seller_label', 'revenue'], ascending=[True, False])

plt.ylabel('Categoria')top_seller_categories = (

plt.tight_layout()

plt.show()seller_category_share['category_share'] = seller_category_share.groupby('seller_label')['revenue'].transform(lambda x: x / x.sum())

seller_category_share = seller_category.copy()

## Eficiência e performance dos sellers

In [ ]:
plt.figure(figsize=(12, 6))
order_size = seller_metrics['order_count'] / seller_metrics['order_count'].max() * 200
sns.scatterplot(
    data=seller_metrics,
    x='avg_order_value',
    y='avg_freight',
    size=order_size,
    sizes=(40, 300),
    palette='coolwarm',
    legend=False,
    alpha=0.7
)
plt.title('Ticket médio x frete médio por seller')
plt.xlabel('Ticket médio por ordem (R$)')
plt.ylabel('Frete médio por ordem (R$)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
low_efficiency = seller_metrics.sort_values('freight_ratio', ascending=False).head(10)
sns.barplot(data=low_efficiency, x='freight_ratio', y='seller_label', palette='mako')
plt.title('Top 10 sellers com maior proporção de frete sobre receita')
plt.xlabel('Frete / Receita')
plt.ylabel('Seller')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
revenue_per_order_sorted = seller_metrics.sort_values('revenue_per_order', ascending=False).head(10)
sns.barplot(data=revenue_per_order_sorted, x='revenue_per_order', y='seller_label', palette='rocket')
plt.title('Top 10 sellers por receita média por pedido')
plt.xlabel('Receita média por pedido (R$)')
plt.ylabel('Seller')
plt.tight_layout()
plt.show()

### Conclusão do escopo de sellers

Este notebook foca em métricas que ajudam a entender os sellers como clientes da Olist e identificar oportunidades de crescimento, eficiência de vendas e performance por categoria.